# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(dplyr)
library(rstatix)

options(tibble.width = Inf)

## Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/siRNA/output"
filter_dapi = TRUE

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/Figures/siRNA/counts.csv")

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
color_condition = c("Control" = "#7BBA56", 
               "Reversine" = "#87549B", 
               "Mosaic" = "#E8973E")

col_Annexin= "#4db2cb"

col_ZVAD = c("ZVAD+" = "#6a6969", 
               "ZVAD-" = "#bbbbbb")

col_GFP = "#7BBA56"
col_RFP = "#cb377c"

col_structure= c("developed" = "#4674b8",  
        "small.cavity" ="#F09938" , 
        "failed" = "#bbbbbb")

## 1. Extract summary files

In [ ]:
merged_df <- read_csv(analysis_summary_files)


In [ ]:
head(merged_df)

In [ ]:
colnames(merged_df)

In [ ]:
unique(merged_df$condition)

In [ ]:
df_sample = merged_df

# Plot 

## Contingency

In [ ]:
head(df_sample)

In [ ]:
df_sample_sub = df_sample %>%
filter(condition %in% c('control', 'ECAD'))

In [ ]:
contingency <- df_sample_sub %>%
  group_by(condition) %>%
  summarise(
    developed = sum(n_developed, na.rm = TRUE),
    failed = sum(n_failed, na.rm = TRUE),
    .groups = "drop"
  )

contingency

In [ ]:
chisq_mat <- contingency %>%
  column_to_rownames("condition") %>%
  as.matrix()

chisq_mat

In [ ]:
chisq_result <- chisq.test(chisq_mat)

chisq_result

In [ ]:
chisq_result$stdres

In [ ]:
melt_data =  melt(contingency, id = c("condition"), measure.vars = c("developed",'failed'),variable.name = "structure", value.name='frequency' )

melt_data$condition = factor(melt_data$condition, levels =unique(melt_data$condition))
head(melt_data)

In [ ]:
df_percent <- melt_data  %>%
  group_by(condition) %>%
  mutate(
    total = sum(frequency),
    percentage = frequency / total * 100
  ) %>%
  ungroup()

df_percent

In [ ]:
title = "proportion developed_contingency"
w <- 1.8
h <- 1.7
options(repr.plot.width=w, repr.plot.height=h)

sample_order = c("failed", "developed")

df_percent<- df_percent%>%
mutate(structure	 = factor(structure	, levels = sample_order))


sample_order2 = c("control", "ECAD")
df_percent<- df_percent%>%
mutate(condition	 = factor(condition, levels = sample_order2))
  
p = ggplot(df_percent, aes(x = condition , y = percentage, fill  = structure)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = "stack", alpha = 0.6, width = 0.8) +   # error bars
    labs(
      title = title,
      y = "proportion developed",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      #legend.position = "none", 
      legend.key.size = unit(0.3, "cm")
      
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
      #facet_wrap( ~ condition_2) +
      scale_fill_manual(values=col_structure)

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
df <- merged_df %>%
  mutate(
    EXP = factor(EXP),
    condition = relevel(factor(condition), ref = "control")
  )

model <- glm(
  cbind(n_developed, n_failed) ~ condition + EXP,
  family = binomial,
  data = df
)

summary(model)

# Overall significance of condition
drop1(model, test = "Chisq")

In [ ]:
df_sample

In [ ]:
# df columns:
# EXP, condition, n_developed, n_failed

tab <- xtabs(
  cbind(n_developed, n_failed) ~ condition + EXP,
  data = df_sample
)

tab

In [ ]:
mantelhaen.test(tab)